# Task D: RL allocation policy
**Responsible:** _(Aldo Patrone)_

A learned allocation policy as a swappable AllocationStrategy, faithful to Middelhuis et al. (2025).

- **MaskablePPO (primary, paper MDP).** State of size 2R+A (per resource a busy flag and its current activity, per activity a normalized queue length), action space of all (resource, activity) pairs plus postpone with action masking, cycle-time reward. Trained via scripts/train_ppo.py (sb3-contrib in .venv-ppo), writing results/ppo_model.zip, results/ppo_comparison.csv and results/rl_learning_curve.png.
- **Auxiliary numpy REINFORCE.** A per-candidate-scoring policy (scripts/train_rl.py), exploratory, serialized to results/rl_policy.json.

Deployment plugs into AllocationStrategy.pick(candidates, context). For MaskablePPO it is a per-task reduction (optimization/ppo_agent.PPOAllocation), with the queue term reduced at inference. The method comparison is in 2.6_evaluation.

In [1]:
import sys, os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
import numpy as np
import matplotlib.pyplot as plt
from resources.log_loader import load_slim_log
from optimization.environment import build_env_config
from optimization.rl_agent import train, RLAllocation

cfg = build_env_config(
    load_slim_log('data/BPI Challenge 2017.xes', 'data/bpic17_slim.parquet'),
    permissions_path='results/permissions_roles.json',
    calendars_path='results/availability_calendars.json',
)
print('env:', len(cfg['activity_mix']), 'activities,', len(cfg['calendars']), 'resources')

[ArrivalEngine] Trained successfully. Base Global Scale: 1002.23s


[ProcessTimeEngine] Loaded models successfully
[BPMNEngine] Model successfully loaded and converted to Petri net.


env: 26 activities, 149 resources


## Auxiliary REINFORCE baseline (short illustration run)
A lightweight numpy REINFORCE baseline. The paper-faithful method is MaskablePPO (state 2R+A, (r,a) action space, cycle-time reward), whose curve results/rl_learning_curve.png and evaluation results/ppo_comparison.csv come from scripts/train_ppo.py. This run saves to a separate file so it does not overwrite that curve.

In [ ]:
net, history = train(cfg, episodes=150, hidden=24, lr=0.05, ep_len=400, seed=1, entropy_beta=0.01)
h = np.array(history)
smooth = np.convolve(h, np.ones(20)/20, mode='valid')
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(h, alpha=0.3, label='episode return')
ax.plot(range(len(smooth)), smooth, label='20-episode moving avg')
ax.set_xlabel('episode'); ax.set_ylabel('return')
ax.set_title('Auxiliary REINFORCE learning curve (numpy baseline)'); ax.legend()
# separate file: the report's Fig. (rl_curve) uses the MaskablePPO curve from scripts/train_ppo.py
fig.tight_layout(); fig.savefig('results/reinforce_learning_curve.png', dpi=120)
print('first-30 avg %.2f -> last-30 avg %.2f' % (h[:30].mean(), h[-30:].mean()))

## Deployable policy
The trained policy (from scripts/train_rl.py) is serialized to results/rl_policy.json and loaded as an AllocationStrategy. Inference is numpy-only. Postpone maps to returning None (the core suspends/retries).

In [3]:
rl = RLAllocation.load('results/rl_policy.json')
class _Ctx:
    load = {'User_1': 5.0, 'User_2': 0.0}
    busy = set()
    class event:
        activity = 'W_Validate application'
print('RLAllocation.pick ->', rl.pick({'User_1', 'User_2', 'User_3'}, _Ctx()))
print('OK: trained RL policy runs via the standard pick(candidates, context) interface')

RLAllocation.pick -> User_1
OK: trained RL policy runs via the standard pick(candidates, context) interface
